# Predicting warranty claims within one year
Target: `claim_within_year`. Follows `checklist.md`.

**Data notes.** 7,000 rows, one per `device_serial` (no repeated entities) and no date column, so there is no time order to respect. A random stratified split is therefore the appropriate structure (checklist items 1-2); whole-entity grouping is automatically satisfied because each serial appears once. `device_serial` is an identifier and is dropped. All other columns (channel, device age, price, reported daily hours) are recorded at sale/registration, so they are known at prediction time.

In [ ]:
import numpy as np, pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_predict, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score, precision_score, recall_score, confusion_matrix
SEED=0
df=pd.read_csv('data.csv')
assert df.device_serial.is_unique
print(df.shape, df.isna().sum().sum(), 'missing'); print(df.claim_within_year.mean())
print((df.daily_hours>24).sum(), 'rows report >24 daily hours (impossible; kept as-is, flagged)')

## 1. Hold out the test set (before any modelling)
20% test, stratified, never touched until the final step.

In [ ]:
FEATURES=['channel','device_age_years','price_eur','daily_hours']
X=df[FEATURES]; y=df.claim_within_year
X_tr,X_te,y_tr,y_te=train_test_split(X,y,test_size=0.2,stratify=y,random_state=SEED)
print(len(X_tr),len(X_te),y_tr.mean(),y_te.mean())

## 2. Candidate models (named before scoring anything)
1. **Baseline**: predicts the training prevalence (`DummyClassifier(prior)`).
2. **Logistic regression** (one-hot channel, standardised numerics), grid over `C` in {0.01, 0.1, 1, 10}.
3. **Gradient boosting** (`HistGradientBoostingClassifier`), grid over `max_depth` {2,3} x `learning_rate` {0.03,0.1}, `max_iter`=200.

Selection metric: CV ROC-AUC; secondary: average precision. Threshold: chosen on out-of-fold predictions of the winner to maximise F1 (claims are ~13%, so 0.5 would be useless). All preprocessing lives inside the pipelines, so it is refit in each CV fold.

In [ ]:
cv=StratifiedKFold(5,shuffle=True,random_state=SEED)
cands={
 'baseline':(Pipeline([('m',DummyClassifier(strategy='prior'))]),{'m__strategy':['prior']}),
 'logreg':(Pipeline([('p',ColumnTransformer([('cat',OneHotEncoder(handle_unknown='ignore'),['channel']),
                                ('num',StandardScaler(),['device_age_years','price_eur','daily_hours'])])),
                     ('m',LogisticRegression(max_iter=1000))]),{'m__C':[0.01,0.1,1,10]}),
 'hgb':(Pipeline([('p',ColumnTransformer([('cat',OneHotEncoder(handle_unknown='ignore'),['channel'])],remainder='passthrough')),
                  ('m',HistGradientBoostingClassifier(max_iter=200,random_state=SEED))]),
        {'m__max_depth':[2,3],'m__learning_rate':[0.03,0.1]}),
}
res={}
for n,(p,g) in cands.items():
    gs=GridSearchCV(p,g,scoring='roc_auc',cv=cv).fit(X_tr,y_tr)
    res[n]=gs; print(n, round(gs.best_score_,4), gs.best_params_)

## 3. Choose model and threshold on CV only

In [ ]:
ap={n:average_precision_score(y_tr,cross_val_predict(g.best_estimator_,X_tr,y_tr,cv=cv,method='predict_proba')[:,1]) for n,g in res.items()}
print('CV average precision:',ap)
best=max((n for n in res if n!='baseline'),key=lambda n:res[n].best_score_)
oof=cross_val_predict(res[best].best_estimator_,X_tr,y_tr,cv=cv,method='predict_proba')[:,1]
ths=np.linspace(0.02,0.6,59); f1s=[f1_score(y_tr,oof>=t) for t in ths]
THR=float(ths[int(np.argmax(f1s))])
print('chosen model:',best,res[best].best_params_,'| threshold',round(THR,3),'| CV F1',round(max(f1s),3))

**Decision (recorded before test scoring):** the model, hyperparameters and threshold printed above are final. Nothing is changed after seeing the test score.

## 4. Refit on all training rows, score the test set once

In [ ]:
final=res[best].best_estimator_.fit(X_tr,y_tr)
p=final.predict_proba(X_te)[:,1]; base=res['baseline'].best_estimator_.fit(X_tr,y_tr).predict_proba(X_te)[:,1]
pred=p>=THR
print(f'TEST ROC-AUC {roc_auc_score(y_te,p):.3f} (baseline {roc_auc_score(y_te,base):.3f})')
print(f'TEST avg precision {average_precision_score(y_te,p):.3f} (prevalence {y_te.mean():.3f})')
print(f'TEST @thr {THR:.2f}: precision {precision_score(y_te,pred):.3f} recall {recall_score(y_te,pred):.3f} F1 {f1_score(y_te,pred):.3f}')
print(confusion_matrix(y_te,pred))
rng=np.random.default_rng(SEED); b=[]
for _ in range(1000):
    i=rng.integers(0,len(y_te),len(y_te))
    if y_te.iloc[i].nunique()>1: b.append(roc_auc_score(y_te.iloc[i],p[i]))
print('ROC-AUC 95% bootstrap CI',np.percentile(b,[2.5,97.5]).round(3))